# Water-Energy-Food nexus and water diplomacy with `wefnexus`

This notebook walks through the `wefnexus` toolkit on the stylised **Azura River** basin, a fictional
transboundary river shared by three riparians ordered upstream to downstream:

* **Highland** - mountainous upstream state, generates most of the runoff, large hydropower, little irrigation;
* **Midland** - mid-stream state with fast-growing cities, thermal power and an expanding irrigation sector;
* **Delta** - arid downstream state with a very large population, the biggest irrigation sector,
  groundwater overdraft and desalination.

All numbers are illustrative (about 28 km3/yr mean annual flow) and are *not* data for any real river.

**What you will see**

1. Water stress indicators (Falkenmark 1989; SDG 6.4.2) and a single-year **basin routing** that serves
   sectors in priority order, respects treaty caps and environmental flows and operates reservoirs
   (a WEAP-style annual balance, Yates et al. 2005).
2. A 25-year **scenario simulation** (baseline, climate change, combined adaptation) coupling water,
   energy (hydropower, thermal cooling, energy for water) and food (FAO-56 crop water requirements and
   the FAO-33 yield response), summarised by composite **WEF security indices** (Hoff 2011).
3. The **water diplomacy** toolbox: hydro-hegemony (Zeitoun & Warner 2006), bankruptcy sharing rules
   (Mianabadi et al. 2014; the Talmud rule of Aumann & Maschler 1985), BATNA / ZOPA negotiation
   analysis, treaties and their institutional resilience (De Stefano et al. 2012), Basins-at-Risk
   events and a conflict risk index (Wolf, Yoffe & Giordano 2003) and Sadoff & Grey (2002) benefit sharing.
4. **Optimisation**: an LP allocation and the efficiency-equity Pareto front ("price of fairness").
5. The basin as a **network** (networkx).

Every figure is produced by `wefnexus.viz`, every table by the core modules - the notebook only glues them together.

## 0. Setup

In **Google Colab** uncomment and run the `pip install` line below (it installs the package straight from
GitHub), then re-run the imports.  In a local clone, `pip install -e .` from the repository root does the same
job; the import cell also falls back to the repository checkout when the package is not installed.

In [ ]:
# In Google Colab, install the package first (uncomment the next line), then run the rest of the notebook:
# !pip install -q git+https://github.com/YousefWEF/YousefWEF.git
#
# From a local clone instead:   pip install -e .      (run in the repository root)
# Optional extras for the figures and tables:   pip install matplotlib networkx pandas

In [ ]:
import math
import sys
from pathlib import Path

try:
    import wefnexus
except ImportError:  # running from a clone that is not installed: add the repository root to sys.path
    _root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
    sys.path.insert(0, str(_root))
    import wefnexus

from wefnexus import diplomacy as D
from wefnexus import optimize as O
from wefnexus import scenarios as S
from wefnexus import sustainability as SU
from wefnexus import viz as V
from wefnexus import water as W
from wefnexus.data import example_basin
from wefnexus.models import SECTOR_PRIORITY, Scenario, Sector
from wefnexus.water import natural_flows, route_basin

YEARS = 25                       # simulation horizon
DROUGHT = 0.6                    # flow factor of a drought year (-40 % natural flow)
OUTPUT_DIR = Path("output")      # CSV and PNG files go here
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def show(rows, columns=None):
    """Print a list of dicts as an aligned text table."""
    print(S.format_table(rows, columns))


print("wefnexus", wefnexus.__version__)

## 1. The basin

A `Basin` is a list of `Riparian` objects ordered upstream to downstream.  Each riparian has sector water
demands (Mm3/yr), an in-stream environmental flow requirement, crops, an energy system, a reservoir,
groundwater and the power pillars used by the hydro-hegemony analysis.  `treaty_allocation_mm3` is the
riparian's treaty entitlement to withdraw surface water.

In [ ]:
basin = example_basin()
print(basin.name, "- natural flow", f"{basin.total_natural_flow():,.0f}", "Mm3/yr, population",
      f"{basin.total_population() / 1e6:.0f}", "million")

rows = []
for r in basin.riparians:
    w = r.demand.withdrawals()
    rows.append({
        "riparian": r.name,
        "population_M": r.population / 1e6,
        "local_inflow_mm3": r.local_inflow_mm3,
        "withdrawal_demand_mm3": sum(w.values()),
        "agricultural_mm3": w[Sector.AGRICULTURAL],
        "env_flow_mm3": r.demand.environmental,
        "reservoir_capacity_mm3": r.reservoir_capacity_mm3,
        "irrigated_ha": r.irrigated_area_ha(),
        "hydro_mw": r.energy.hydropower_capacity_mw,
        "treaty_mm3": r.treaty_allocation_mm3,
    })
show(rows)

## 2. Water stress indicators

Classic per-capita and use-to-availability indicators, computed from the *natural* inflow of each reach
(what arrives with zero upstream use) plus local groundwater recharge:

* **Falkenmark** per-capita renewable water (m3/person/yr): > 1700 no stress, 1000-1700 stress,
  500-1000 scarcity, < 500 absolute scarcity.
* **Water exploitation index** = withdrawal / renewable water (0.2 moderate, 0.4 severe stress).
* **SDG 6.4.2** water stress = 100 x withdrawal / (renewable - environmental flow), in percent.

In [ ]:
flows = natural_flows(basin)
rows = []
for r in basin.riparians:
    renewable = flows[r.name] + r.groundwater_recharge_mm3
    withdrawal = r.demand.total_withdrawal() + r.groundwater_abstraction_mm3
    pc = W.per_capita_water(renewable, r.population)
    rows.append({
        "riparian": r.name,
        "natural_inflow_mm3": flows[r.name],
        "per_capita_m3": pc,
        "falkenmark": W.falkenmark_category(pc),
        "WEI": W.water_exploitation_index(withdrawal, renewable),
        "sdg_6_4_2_pct": W.sdg_642_water_stress(withdrawal, renewable, r.demand.environmental),
        "groundwater_stress": W.groundwater_stress(r.groundwater_abstraction_mm3, r.groundwater_recharge_mm3),
        "dependency_ratio": W.dependency_ratio(flows[r.name] - r.local_inflow_mm3, renewable),
    })
show(rows)

## 3. One year of basin routing

`route_basin` walks the reaches from upstream to downstream.  In each reach the inflow (upstream outflow
plus local runoff) and the reservoir storage are used to serve the sectors in priority order (municipal,
industrial, energy, agricultural), subject to the treaty cap on surface withdrawal and to the environmental
flow that must leave the reach; part of the surplus refills the reservoir.  The exact mass balance
`inflow + storage_start = consumption + outflow + storage_end + evaporation` holds in every reach.

In [ ]:
balances = {"normal": route_basin(basin), "drought": route_basin(basin, DROUGHT)}
for label, bal in balances.items():
    print(f"\n{label} year: natural flow {bal.natural_flow_mm3:,.0f} Mm3, outflow to sea "
          f"{bal.outflow_to_sea_mm3:,.0f} Mm3, basin supply ratio {bal.supply_ratio():.3f}, "
          f"max mass-balance error {bal.mass_balance_error():.1e} Mm3")
    show([{
        "riparian": x.name, "inflow_mm3": x.inflow_mm3, "withdrawal_mm3": x.total_withdrawal(),
        "non_river_mm3": x.non_river_supply_mm3, "deficit_mm3": x.total_deficit(),
        "supply_ratio": x.supply_ratio(), "outflow_mm3": x.outflow_mm3,
        "env_flow_mm3": x.environmental_flow_mm3, "env_met": x.env_flow_met, "storage_end_mm3": x.storage_end_mm3,
    } for x in bal.reaches])

## 4. Scenarios over the horizon

The scenario library in `wefnexus.scenarios` provides storylines (`baseline`, `climate_change`, `growth`,
`efficiency`, `unilateral`, `cooperative`, `combined_stress`, `combined_adaptation`).  A `Scenario` scales
natural flow, population, demands, irrigated area, irrigation efficiency and the renewable share year by
year, chooses the sharing arrangement (`allocation_rule`, fixed treaty volumes by default) and can inject
drought years.  `run_scenarios` simulates each one on a fresh copy of the basin; `comparison_table`
condenses the results into one row per scenario and riparian plus a basin row.

In [ ]:
scenarios = [S.get_scenario(name, years=YEARS) for name in ("baseline", "climate_change", "combined_adaptation")]
for sc in scenarios:
    print(f"- {sc.name}: {sc.description}")

results = S.run_scenarios(basin, scenarios)
rows = S.comparison_table(results)
show(rows, ["scenario", "riparian", "mean_supply_ratio", "min_supply_ratio", "supply_reliability",
            "mean_nexus_index", "mean_water_security", "mean_energy_security", "mean_food_security",
            "years_env_flow_unmet", "total_hydropower_gwh", "food_self_sufficiency", "final_storage_mm3",
            "equity_index"])
S.write_table_csv(rows, OUTPUT_DIR / "scenario_comparison.csv")
for name, res in results.items():
    res.to_csv(OUTPUT_DIR / f"records_{name}.csv")
    assert max(b.mass_balance_error() for b in res.balances) < 1e-6
print("\nCSV files written to", OUTPUT_DIR.resolve())

### Supply ratio and WEF security indices over time

`plot_supply_ratio` draws the annual supply ratio (supplied / demanded) of each riparian; `plot_nexus_indices`
shows the water, energy and food security indices and their geometric mean, the WEF nexus index.  All
`wefnexus.viz` functions return the matplotlib `Figure` and accept `save=<path>`.

In [ ]:
res = results["combined_adaptation"]
fig = V.plot_supply_ratio(res, save=OUTPUT_DIR / "supply_ratio.png")

In [ ]:
fig = V.plot_nexus_indices(res, save=OUTPUT_DIR / "nexus_indices.png")

### Water balance of the downstream riparian

Stacked withdrawals by sector (plus non-river supply and unmet demand, so the bar height is the gross
demand), reach inflow, outflow and the environmental-flow requirement, and the reservoir storage below.

In [ ]:
fig = V.plot_water_balance(res, "Delta", basin=basin, save=OUTPUT_DIR / "water_balance_Delta.png")

## 5. Sustainability assessment

`sustainability.assess` turns a simulation into a report: horizon means of the four indices per riparian,
Hashimoto (1982) reliability / resilience / vulnerability of supply, SDG-style indicators (6.4.1 water-use
efficiency, 6.4.2 water stress, 6.5.2 transboundary cooperation, 7.2 renewable share, 2.1 food
self-sufficiency) and basin aggregates including an equity index (1 - Gini of the riparian supply ratios).

In [ ]:
for name, r in results.items():
    report = SU.assess(r)
    print(report.summary())
    print()

In [ ]:
fig = V.plot_nexus_radar(results["baseline"].summary(), save=OUTPUT_DIR / "nexus_radar_baseline.png")

## 6. Water diplomacy I: power and negotiation

**Hydro-hegemony** (Zeitoun & Warner 2006) scores the four pillars of riparian power - geographic position,
material, bargaining and ideational power - on 0..1.  `negotiate` frames a basin year as a **claims
problem**: the estate is the natural flow net of the in-stream requirement at the basin outlet (`bankruptcy_estate`), the claims are the treaty entitlements
(or demands), a bankruptcy rule divides the estate, and each riparian compares its award with its **BATNA**
- what it would get under unilateral, upstream-priority use.  A **zone of possible agreement** exists when
every riparian prefers the proposal.

In [ ]:
rows = []
for r in basin.riparians:
    h = D.hydro_hegemony(r, basin)
    rows.append({"riparian": r.name, **{k: h[k] for k in ("geographic", "material", "bargaining", "ideational", "score")}})
show(rows)
print("power asymmetry:", round(D.power_asymmetry(basin), 3))

neg = D.negotiate(basin, rule="talmud")
print(f"\nestate {neg['estate']:,.0f} Mm3/yr, claims {sum(neg['claims'].values()):,.0f} Mm3/yr, rule {neg['rule']}")
show([{"riparian": n, "claim": neg["claims"][n], "proposal": neg["proposal"][n],
       "satisfaction": neg["satisfaction"][n], "batna": neg["batna"][n], "acceptable": neg["acceptable"][n]}
      for n in basin.names()])
print("ZOPA:", neg["zopa"], "| Gini of proposal:", round(neg["gini"], 3))

## 7. Water diplomacy II: comparing sharing rules

`compare_allocation_rules` applies every rule of `wefnexus.allocation` - proportional, constrained equal
awards (CEA), constrained equal losses (CEL), Talmud, adjusted proportional, equal split and upstream
priority - to the same claims problem, then routes each award vector through the basin as entitlements to
see what is actually delivered.  Compare the normal and the drought year.

In [ ]:
comparisons = {"normal": D.compare_allocation_rules(basin, 1.0), "drought": D.compare_allocation_rules(basin, DROUGHT)}
for label, cmp in comparisons.items():
    print(label, "year")
    show([{"rule": rule, **{f"award_{n}": row["awards"][n] for n in basin.names()}, "gini": row["gini"],
           "min_satisfaction": row["min_satisfaction"], "routed_supply_ratio": row["supply_ratio"],
           "outflow_to_sea": row["outflow_to_sea_mm3"]} for rule, row in cmp.items()])
    print()
fig = V.plot_allocation_rules(comparisons["drought"], title="Allocation rules compared - drought year",
                              save=OUTPUT_DIR / "allocation_rules_drought.png")

## 8. Water diplomacy III: treaties, events and conflict risk

A `Treaty` records fixed entitlements and institutional features; `treaty_resilience` scores them after
De Stefano et al. (2012) and `treaty_compliance` checks a routed year against the entitlements.
`BarEvent`s are interactions on the Basins-at-Risk scale (-7 war ... +7 unification); `cooperation_index`,
`conflict_intensity` and the TWINS matrix (Mirumachi & Allan 2007) summarise them.  `conflict_risk_index`
combines water stress, downstream dependency, power asymmetry, institutional weakness, conflict history,
hydrological variability, unmet environmental flows and upstream dam filling into one 0..1 score.

In [ ]:
treaty = D.treaty_from_basin(basin, name="Azura River Water Agreement", data_sharing=True, joint_institution=True,
                             dispute_resolution=True, review_period_years=10, year_signed=2010)
print(treaty.name, "- parties", treaty.parties, "- resilience", round(D.treaty_resilience(treaty), 2))
show([{"party": p, **{k: c[k] for k in ("entitlement", "withdrawal", "compliance_ratio", "delivery_ratio", "delivered")}}
      for p, c in D.treaty_compliance(treaty, balances["drought"]).items()])

events = [
    D.BarEvent(2013, ["Highland", "Midland"], -2, "infrastructure", "Midland objects to the Highland dam filling plan"),
    D.BarEvent(2016, ["Highland", "Midland", "Delta"], 3, "data", "Joint hydrological monitoring programme"),
    D.BarEvent(2019, ["Midland", "Delta"], -3, "quantity", "Delta recalls its envoy over dry-season deliveries"),
    D.BarEvent(2022, ["Highland", "Midland", "Delta"], 4, "treaty", "Basin commission and drought protocol agreed"),
]
coop, conf = D.cooperation_index(events), D.conflict_intensity(events)
print(f"\ncooperation index {coop:.2f}, conflict intensity {conf:.2f}, TWINS class {D.twins_classification(coop, conf)}")

risks = {
    "normal year, treaty": D.conflict_risk_index(basin, balances["normal"], treaty, events),
    "drought year, treaty": D.conflict_risk_index(basin, balances["drought"], treaty, events),
    "drought year, no treaty": D.conflict_risk_index(basin, balances["drought"], None, events),
}
show([{"case": k, "score": v["score"], "category": v["category"], **v["components"]} for k, v in risks.items()])

## 9. Benefit sharing

Sadoff & Grey (2002) distinguish benefits *to* the river (environmental flows), *from* the river (water
value and hydropower), *because of* the river (reduced conflict costs) and *beyond* the river (catalytic
gains of cooperation).  Here the unilateral upstream-priority year is compared with the Talmud proposal
routed as entitlements.

In [ ]:
unilateral = route_basin(basin, entitlements={n: None for n in basin.names()})
cooperative = route_basin(basin, entitlements=neg["proposal"])
matrix = D.benefit_sharing_matrix(basin, unilateral, cooperative, treaty=treaty, events=events)
show([{"riparian": n, "to_the_river": m["to_the_river"], "from_the_river_bn_usd": m["from_the_river"] / 1e9,
       "because_of_the_river": m["because_of_the_river"], "beyond_the_river_bn_usd": m["beyond_the_river"] / 1e9,
       "hydropower_gwh": m["hydropower_gwh"]} for n, m in matrix.items()])

## 10. Optimisation and the price of fairness

`optimize_allocation` solves a linear programme for the basin-wide withdrawals that maximise economic
benefit (or the minimum supply ratio, or a weighted mix) subject to the river routing, environmental flows and
optional minimum-supply guarantees.  `pareto_front` sweeps the guaranteed minimum supply ratio with the
epsilon-constraint method (Haimes et al. 1971) and reveals the **price of fairness** (Bertsimas et al. 2011):
how much total benefit the basin gives up to protect its worst-off riparian.

In [ ]:
best = O.optimize_allocation(basin, DROUGHT, objective="benefit")
print(best.summary())
front = O.pareto_front(basin, DROUGHT, points=11)
show([{"min_supply_ratio": p["min_supply_ratio"], "benefit_bn_usd": p["total_benefit_usd"] / 1e9, "gini": p["gini"],
       "achieved_min_ratio": p["achieved_min_supply_ratio"], "mean_supply_ratio": p["mean_supply_ratio"]} for p in front])
fig = V.plot_pareto(front, title="Efficiency-equity trade-off - drought year", save=OUTPUT_DIR / "pareto_front.png")

## 11. The nexus as a network

`nexus_graph` builds a directed graph: the river chain from headwater to sea through the riparians, the four
sector nodes of each riparian (edge weight = withdrawal, Mm3/yr) and the non-river sources (local runoff,
groundwater, desalination).  Pass a routed `BasinBalance` to weight the edges with actual flows.

In [ ]:
G = V.nexus_graph(basin)
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
print("river chain:", [(u, v, round(d["weight"])) for u, v, d in G.edges(data=True) if d["kind"] == "river"])
fig = V.plot_nexus_graph(basin, balance=balances["drought"], save=OUTPUT_DIR / "nexus_graph_drought.png")

## 12. Build your own scenario

A `Scenario` is a plain dataclass: combine your own drivers (stochastic flows, demand growth, a different
sharing rule ...) and run it with `wefnexus.nexus.run_nexus` or `scenarios.run_scenarios`.  Replace
`example_basin()` with your own `Basin` (same dataclasses) to study a real river.

In [ ]:
from wefnexus.nexus import run_nexus

custom = Scenario(name="my_stochastic_cea", years=YEARS, flow_change_pct_by_end=-25, population_growth_rate=0.02,
                  demand_growth_rate=0.02, irrigation_efficiency_target=0.65, allocation_rule="cea",
                  stochastic=True, seed=7, description="stochastic flows, growth, efficiency, CEA sharing")
custom_result = run_nexus(basin, custom)
summary = custom_result.summary()
show([{"riparian": n, **{k: summary[n][k] for k in ("supply_ratio", "min_supply_ratio", "supply_reliability",
                                                     "nexus_index", "food_self_sufficiency")}}
      for n in custom_result.riparian_names() + ["BASIN"]])
fig = V.plot_supply_ratio(custom_result)

## References

* Aumann, R. & Maschler, M. (1985). Game theoretic analysis of a bankruptcy problem from the Talmud. *J. Economic Theory* 36.
* Bertsimas, D., Farias, V. & Trichakis, N. (2011). The price of fairness. *Operations Research* 59.
* De Stefano, L. et al. (2012). Climate change and the institutional resilience of international river basins. *J. Peace Research* 49.
* Falkenmark, M. (1989). The massive water scarcity now threatening Africa. *Ambio* 18.
* Hashimoto, T., Stedinger, J. & Loucks, D. (1982). Reliability, resiliency and vulnerability criteria. *Water Resources Research* 18.
* Hoff, H. (2011). *Understanding the Nexus*. Bonn 2011 Nexus Conference, SEI.
* Mianabadi, H. et al. (2014). A new bankruptcy method for conflict resolution in water resources allocation. *J. Environmental Management* 144.
* Mirumachi, N. & Allan, J. A. (2007). Revisiting transboundary water governance: the TWINS framework.
* Sadoff, C. & Grey, D. (2002). Beyond the river: the benefits of cooperation on international rivers. *Water Policy* 4.
* Wolf, A., Yoffe, S. & Giordano, M. (2003). International waters: identifying basins at risk. *Water Policy* 5.
* Zeitoun, M. & Warner, J. (2006). Hydro-hegemony: a framework for analysis of trans-boundary water conflicts. *Water Policy* 8.